# BCS Determination - Visualization of Results

This notebook demonstrates how to load a trained PyTorch Lightning model and visualize its predictions on sample images.

In [ ]:
import os
import glob
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

from src.bcs_pipeline.lightning_module.bcs_determination_module import LitBcsDetermination
from src.bcs_pipeline.data.stanford_bcs_datamodule import StanfordBcsDataModule

## 1. Configuration
Update these paths according to your environment and training setup.

In [ ]:
CHECKPOINT_PATH = "path/to/your/checkpoint.ckpt"
DATA_DIR = "/home/SPeillet/Downloads/stanford_bcs/images"
MODEL_NAME = "resnet50"
NUM_CLASSES = 120

## 2. Load Model and Transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

try:
    model = LitBcsDetermination.load_from_checkpoint(
        checkpoint_path=CHECKPOINT_PATH,
        model_name=MODEL_NAME,
        num_classes=NUM_CLASSES,
        map_location=device
    )
    model.to(device)
    model.eval()
    print("Model loaded successfully!")
except Exception as e:
    print(f"Could not load model, ensure CHECKPOINT_PATH is correct. Error: {e}")
    model = None

In [ ]:
# Load transforms
datamodule = StanfordBcsDataModule(data_dir=DATA_DIR, image_size=224)
val_transforms = datamodule.val_transforms

# Load class names
def load_class_names(data_dir):
    images_dir = os.path.join(data_dir, "Images")
    if not os.path.exists(images_dir):
        return []
    classes = [d.name for d in os.scandir(images_dir) if d.is_dir()]
    classes.sort()
    clean_classes = ["-".join(c.split("-")[1:]) for c in classes]
    return clean_classes

class_names = load_class_names(DATA_DIR)
print(f"Loaded {len(class_names)} classes.")

## 3. Visualization Function

In [ ]:
def predict_and_visualize(image_path, model, transform, class_names, device):
    if not model:
        print("No model loaded.")
        return
        
    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        top_prob, top_class = torch.max(probs, dim=1)
        
        top_prob = top_prob.item()
        top_class = top_class.item()
        
    pred_label = class_names[top_class] if class_names and top_class < len(class_names) else f"Class {top_class}"
    
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Pred: {pred_label} ({top_prob*100:.1f}%)", fontsize=14)
    plt.axis("off")
    plt.show()

## 4. Test on Random Samples

In [ ]:
if os.path.exists(os.path.join(DATA_DIR, "Images")):
    all_images = glob.glob(os.path.join(DATA_DIR, "Images", "*", "*.jpg"))
    if all_images:
        sample_images = random.sample(all_images, min(3, len(all_images)))
        for img_path in sample_images:
            predict_and_visualize(img_path, model, val_transforms, class_names, device)
    else:
        print("No images found in dataset.")
else:
    print("Dataset directory not found. Please provide a valid DATA_DIR.")